# Ollama+LangChain: Models, Prompts, LCEL, Parser

LangChain is an open-source framework designed to simplify the creation of applications using Large Language Models (LLMs). It acts as an orchestration layer, enabling developers to connect LLMs to external data sources, manage memory, and build complex, multi-step workflows (chains).

## 1. LLM Model

- First we need to configure Ollama port

In [6]:
import os
# get home and job id
home_dir =  os.getenv('HOME')
job_id = os.getenv('SLURM_JOB_ID')

# get ollama directory or default to home
ollama_dir = os.getenv('OLLAMA_BASE_DIR', home_dir)

try:    
    with open(f"{ollama_dir}/ollama/host_{job_id}.txt") as f:
        HOST = f.read().strip()
    with open(f"{ollama_dir}/ollama/port_{job_id}.txt") as f:
        PORT = f.read().strip()
    ollama_url = f"http://{HOST}:{PORT}"
except Exception as e:
    print("[⚠️] Could not read host/port, you manually set the `ollama_url` below.")

print("The port that serve ollama is: ", ollama_url)


The port that serve ollama is:  http://bcm-dgxa100-0001:38833


- Next let define chat model

In [10]:
from langchain_ollama import ChatOllama
llm = ChatOllama(base_url=ollama_url,
                model="gemma3:4b",  
                temperature=0.5, 
                num_predict=256)

In [11]:
response = llm.invoke("Write a limerick about SMU Dallas")
print(response.content)

Okay, here's a limerick about SMU Dallas:

In Dallas, SMU stands tall,
With Mustangs, answering freedom’s call.
A campus so grand,
Across the whole land,
A prestigious school for one and all. 

---

Would you like me to try another one, or perhaps a different style?


## 2. Prompts
### 2.1 PromptTemplate
- LLM treats role differently between system # users # assistant


In [12]:
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate(
        input_variables=["topic","level"],
        template="Explain {topic} for {level} in simple term"
)

# Format it with actual values
formatted_prompt = prompt.format(    
    topic="neural networks",
    level="beginner"
)
formatted_prompt

'Explain neural networks for beginner in simple term'

In [13]:
print(llm.invoke(formatted_prompt).content)

Okay, let's break down neural networks in a way that's easy to understand, even if you've never heard of them before.

**1. The Basic Idea: Mimicking the Brain**

Imagine your brain. It's made up of billions of tiny cells called neurons, all connected to each other. When you learn something – like recognizing a cat – your brain doesn't do it with a specific set of rules. Instead, it learns by adjusting the connections between these neurons.

Neural networks are *inspired* by this. They're computer programs designed to learn in a similar way.

**2. The Building Blocks: Neurons (Nodes)**

* **Nodes:** These are the basic units of a neural network. Think of them like individual neurons in your brain.
* **Inputs:** Each node receives information from other nodes or directly from the data you're feeding it.
* **Weights:** Each connection between nodes has a "weight" associated with it. This weight determines how important that connection is.  A high weight means that connection has a big in

### 2.2 ChatPromptTemplate
- Using **from_template** for quick template

In [15]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("Explain {topic} for {level} in simple term")
response = llm.invoke(prompt.format(topic="neural networks",level="beginner"))
print(response.content)


Okay, let's break down neural networks in a way that's easy to understand, even if you've never heard of them before.

**1. The Basic Idea: Mimicking the Brain**

Imagine your brain. It's made up of billions of tiny cells called neurons, all connected and working together. Neural networks are *inspired* by this. They're a way to create computer programs that can learn and make decisions, just like our brains do.

**2. Building Blocks: Neurons (Nodes)**

* **Think of a neuron as a tiny decision-maker.** It receives information, does a simple calculation, and then passes the result on.
* **Input:** A neuron gets information from other neurons (or directly from the data you're feeding it).
* **Weights:** Each connection between neurons has a "weight." This weight determines how important that piece of information is.  Think of it like volume control - some inputs are louder (more important) than others.
* **Calculation:** The neuron multiplies each input by its corresponding weight, adds 

- Using **from_messages** for more control template

In [16]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI tutor"),
    ("user", "Explain {topic} for a {level} level student.")
])

# Format the prompt with variables
formatted_prompt = prompt.format_messages(
    topic="neural networks",
    level="beginner"
)

In [17]:
print(llm.invoke(formatted_prompt).content)

Okay, let's break down neural networks! Don't worry, it sounds complicated, but the basic idea is actually pretty cool and surprisingly intuitive. 

**1. The Brain Inspiration:**

The whole concept of neural networks comes from how our brains work.  Your brain is made up of billions of tiny cells called neurons, which are connected to each other. When you learn something – like recognizing a cat – these connections change, making it easier for your brain to recognize cats in the future.

**2. What is a Neural Network?**

A neural network is a computer program designed to mimic this process. It's made up of interconnected “nodes” (like neurons) organized in layers. Here’s a simplified breakdown:

* **Input Layer:** This is where you feed the network information. Think of it like your eyes seeing a picture of a cat. The input layer receives raw data – in this case, the pixel values of the image.
* **Hidden Layers:** These are the layers in between the input and output layers. This is whe

### 3. LCEL - Basic
- **LCEL**: **LangChain Expression Language** is the modern way to compose chains in LangChain using the pipe operator **(|)**. It's like building a pipeline where data flows through different components.
- it works with any prompt from PromptTemplate or ChatPromptTemplate

In [18]:
chain = prompt | llm

# Invoke
response = chain.invoke({    
    "topic": "neural networks",
    "level": "beginner"
})

# Output
print(response.content)


Okay, let’s break down neural networks! Don’t worry, it sounds complicated, but the basic idea is actually pretty cool and surprisingly intuitive.

**1. The Brain Inspiration:**

Think about how your brain works. It’s made up of billions of tiny cells called neurons, all connected to each other. When you learn something, these connections get stronger or weaker. That's the core idea behind neural networks – we're trying to mimic this process.

**2. What is a Neural Network?**

A neural network is a computer program designed to learn patterns, just like your brain. It’s made up of interconnected “nodes” (like neurons) organized in layers.  Here's a simplified breakdown:

* **Input Layer:** This is where you feed the network information.  Think of it like your eyes or ears – it receives the initial data.  For example, if you're trying to teach a network to recognize pictures of cats, the input layer would receive the pixel data from the image.

* **Hidden Layers:** These are the layers i

## 4. Parser
- Langchain Parser tells the model exactly how to format output
- Validate the response
- Convert it into a Python object
### 4.1 StrOutputParser
- **StrOutputParser** simply give you clean text as output so you do not have to append **.content** to the output


In [20]:
from langchain_core.output_parsers import StrOutputParser

# With StrOutputParser
chain = prompt | llm | StrOutputParser()
response = chain.invoke({    
    "topic": "neural networks",
    "level": "beginner"
})
print(response)


Okay, let’s break down neural networks in a way that’s easy to understand. Think of it like teaching a computer to recognize a picture of a cat! 

**1. The Basic Idea: Mimicking the Brain**

Neural networks are inspired by how the human brain works. Our brains are made up of billions of tiny cells called neurons that are connected to each other. When we learn something, these connections get stronger or weaker.

Neural networks try to mimic this process with computer code. They’re designed to learn patterns from data, just like we do.

**2. The Building Blocks: Neurons (or Nodes)**

* **What they are:**  A neural network is made up of lots of interconnected "neurons" (also called nodes).  Think of each neuron as a tiny decision-maker.
* **How they work:**
    * **Input:** Each neuron receives information from other neurons. This information is like a signal.
    * **Weight:** Each connection between neurons has a “weight” assigned to it. This weight determines how important that connec

### 4.2 Pydantic Output Parser
- If you need to format the output, for example in JSON format, you can use **PydanticOutputParser** 

In [21]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List


In [26]:
# Important: Define a structured output format
class SummaryResult(BaseModel):
    concept: str = Field(description="The concept being explained")
    beginner_explanation: str = Field(description="A simple explanation for beginners")
    real_world_analogy: str = Field(description="An easy analogy from daily life")
    key_terms: List[str] = Field(description="3-5 important terms for this concept")

In [27]:
# Build parser + prompt
parser = PydanticOutputParser(pydantic_object=SummaryResult)

prompt = PromptTemplate(
    template="""
Explain {topic} for a {level} student.
Focus on clarity and beginner-friendly language.

{format_instructions}
""",
    input_variables=["topic", "level"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    },
)


In [28]:
# Run chain
chain = prompt | llm | parser

result = chain.invoke({
    "topic": "neural networks",
    "level": "beginner"
})

print(result)
print(type(result))


concept='Neural Networks' beginner_explanation="Imagine you're teaching a computer to recognize pictures of cats. A neural network is like a simplified version of how your brain learns. It's a system made of interconnected 'neurons' that learn from examples.  Instead of being explicitly told 'cats have pointy ears and whiskers,' the network looks at thousands of cat pictures and gradually figures out what patterns make a cat a cat. It adjusts its connections (like strengthening some pathways in your brain) to become better and better at recognizing them." real_world_analogy="Think of it like learning to ride a bike. You don't start knowing exactly how to balance. You fall a lot, and each time you adjust your movements slightly – leaning left, steering a bit – based on feedback (feeling yourself wobble).  A neural network does something similar: it makes small adjustments based on the 'feedback' of whether it's getting the answer right or wrong until it learns to correctly identify patt

In [31]:
result

SummaryResult(concept='Neural Networks', beginner_explanation="Imagine you're teaching a computer to recognize pictures of cats. A neural network is like a simplified version of how your brain learns. It's a system made of interconnected 'neurons' that learn from examples.  Instead of being explicitly told 'cats have pointy ears and whiskers,' the network looks at thousands of cat pictures and gradually figures out what patterns make a cat a cat. It adjusts its connections (like strengthening some pathways in your brain) to become better and better at recognizing them.", real_world_analogy="Think of it like learning to ride a bike. You don't start knowing exactly how to balance. You fall a lot, and each time you adjust your movements slightly – leaning left, steering a bit – based on feedback (feeling yourself wobble).  A neural network does something similar: it makes small adjustments based on the 'feedback' of whether it's getting the answer right or wrong until it learns to correct